# 00 · Environment setup
#
# One-time setup for the GEETA 4B notebook stack.
# Creates `.venv/` in the repo root and installs `requirements.txt`.
#
# **Run from the repo root:**
# ```bash
# jupyter nbconvert --to notebook --execute notebooks/00_environment_setup.ipynb \
#     --output notebooks/00_environment_setup.ipynb --ExecutePreprocessor.timeout=600
# ```
# (or open in JupyterLab and run all cells)


In [ ]:
import json, pathlib, platform, sys, subprocess

ROOT = pathlib.Path.cwd()
print(f"Python      : {sys.version.split()[0]}  ({platform.python_implementation()})")
print(f"OS          : {platform.system()} {platform.release()} ({platform.machine()})")
print(f"Repo root   : {ROOT}")
print(f"Has brief   : {(ROOT/'GEETA_PROJECT_BRIEF.md').exists()}")
print(f"Has config  : {(ROOT/'config'/'config.yaml').exists()}")


In [ ]:
VENV = ROOT / ".venv"
REQ  = ROOT / "requirements.txt"

def run(cmd, **kw):
    print(f"$ {' '.join(cmd)}")
    return subprocess.run(cmd, capture_output=True, text=True, timeout=600, **kw)

if not (VENV / "bin" / "python").exists():
    r = run([sys.executable, "-m", "venv", str(VENV)], timeout=300)
    assert r.returncode == 0, r.stderr
print("venv python:", (VENV / "bin" / "python").resolve())

r = run([str(VENV / "bin" / "pip"), "install", "--quiet", "--upgrade", "pip"])
assert r.returncode == 0, r.stderr

if REQ.exists():
    r = run([str(VENV / "bin" / "pip"), "install", "--quiet", "-r", str(REQ)])
    assert r.returncode == 0, r.stderr
    print("requirements installed OK")
else:
    print("WARNING: requirements.txt not found — expected at repo root")


In [ ]:
# Smoke test: every package the pipeline depends on
import importlib
core = ["pydantic", "pypdfium2", "openpyxl", "PIL", "sympy",
        "reportlab", "matplotlib", "requests", "nbformat"]
missing = []
for m in core:
    try:
        mod = importlib.import_module(m)
        print(f"  ✓ {m:<12} {getattr(mod, '__version__', '')}")
    except ImportError as e:
        missing.append(m)
        print(f"  ✗ {m:<12} MISSING — {e}")
assert not missing, f"missing packages: {missing}"
print("\nENVIRONMENT READY")


In [ ]:
# Optional heavy dependencies (fine-tuning, M4 on the M4 Max).
# These are large downloads; only install when ready to train.
#   .venv/bin/pip install mlx mlx-lm mlx-vlm transformers
print("Next: run notebooks/01_acara_skill_map_and_problem_bank.ipynb from the repo root.")